In [1]:
!pip install pypandoc


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import json
import pandas as pd

df=pd.read_csv("cleaned_all_beauty.csv")
df.head()
df.shape

(25731, 17)

In [2]:
df = df[["rating", "review_text"]].copy()

In [3]:
df

,rating,review_text
0,5.0,This is perfect for my between salon visits. I...
1,5.0,I get Keratin treatments at the salon at least...
2,5.0,This is a really nice moisturizing lotion. It ...
3,3.0,I try to get Keratin treatments every 3 months...
4,5.0,"Really nice small brush. Made well, nice wood ..."
...,...,...
25726,5.0,[[VIDEOID:f4d7a470f32551a5541e089ba23085fb]] I...
25727,5.0,[[VIDEOID:1552f5dc4ee822d5d7a90ac9288ba72c]] O...
25728,5.0,Classic black polish with not much more to tel...
25729,2.0,On the picture it looks like milky half transp...


In [4]:
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df = df.dropna(subset=["rating"])
df["rating"] = df["rating"].astype(int)

In [5]:
df

,rating,review_text
0,5,This is perfect for my between salon visits. I...
1,5,I get Keratin treatments at the salon at least...
2,5,This is a really nice moisturizing lotion. It ...
3,3,I try to get Keratin treatments every 3 months...
4,5,"Really nice small brush. Made well, nice wood ..."
...,...,...
25726,5,[[VIDEOID:f4d7a470f32551a5541e089ba23085fb]] I...
25727,5,[[VIDEOID:1552f5dc4ee822d5d7a90ac9288ba72c]] O...
25728,5,Classic black polish with not much more to tel...
25729,2,On the picture it looks like milky half transp...


In [6]:
df.dropna(subset=["rating", "review_text"], inplace=True)

In [7]:
df["rating"].value_counts()

rating
5    15513
4     4723
3     2556
1     1694
2     1245
Name: count, dtype: int64

In [8]:
def get_sentiment_label(rating):
    if rating in [1, 2]:
        return 0   # negative
    elif rating == 3:
        return 1   # neutral
    else:
        return 2   # positive

df["sentiment"] = df["rating"].apply(get_sentiment_label)


In [9]:
print(df["sentiment"].value_counts())


sentiment
2    20236
0     2939
1     2556
Name: count, dtype: int64


In [10]:
df.head()

,rating,review_text,sentiment
0,5,This is perfect for my between salon visits. I...,2
1,5,I get Keratin treatments at the salon at least...,2
2,5,This is a really nice moisturizing lotion. It ...,2
3,3,I try to get Keratin treatments every 3 months...,1
4,5,"Really nice small brush. Made well, nice wood ...",2


In [11]:
df = df[["sentiment", "review_text"]].copy()

In [12]:
df.head()

,sentiment,review_text
0,2,This is perfect for my between salon visits. I...
1,2,I get Keratin treatments at the salon at least...
2,2,This is a really nice moisturizing lotion. It ...
3,1,I try to get Keratin treatments every 3 months...
4,2,"Really nice small brush. Made well, nice wood ..."


In [13]:
import tensorflow

In [14]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [15]:
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)

In [16]:
print(train_data.shape)
print(test_data.shape)

(20584, 2)
(5147, 2)


In [17]:
# Tokenize text data
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(train_data["review_text"])
X_train = pad_sequences(tokenizer.texts_to_sequences(train_data["review_text"]), maxlen=200)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data["review_text"]), maxlen=200)

In [18]:
print(X_train)

[[   0    0    0 ...   20  197  862]
 [   0    0    0 ...  943  329   17]
 [   0    0    0 ...   20  267    4]
 ...
 [   0    0    0 ...   13    9 2127]
 [   0    0    0 ...  124    4   83]
 [   0    0    0 ...  295   41 1347]]


In [19]:
print(X_test)

[[  15    9   81 ...  109    7  314]
 [   0    0    0 ...   36 1265   58]
 [  10    9 1202 ...   77    1  262]
 ...
 [   0    0    0 ...    5   87   74]
 [   0    0    0 ...  854   10  329]
 [   0    0    0 ...   15    5  575]]


In [20]:
Y_train = train_data["sentiment"]
Y_test = test_data["sentiment"]

In [21]:
print(Y_train)

23409    2
18962    0
22791    2
24883    2
11879    2
        ..
21575    2
5390     2
860      2
15795    1
23654    2
Name: sentiment, Length: 20584, dtype: int64


##LSTM - Long Short-Term Memory

In [22]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input

model = Sequential([
    Input(shape=(200,)),
    Embedding(input_dim=5000, output_dim=128),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(3, activation="softmax")
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 200, 128)            │         640,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 128)                 │         131,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 3)                   │             387 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 771,971 (2.94 MB)

 Trainable params: 771,971 (2.94 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
# compile the model
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

#**Training the Model**

In [24]:
model.fit(X_train, Y_train, epochs=5, batch_size=64, validation_split=0.2)

Epoch 1/5
258/258 ━━━━━━━━━━━━━━━━━━━━ 185s 689ms/step - accuracy: 0.8032 - loss: 0.5569 - val_accuracy: 0.8178 - val_loss: 0.4807
Epoch 2/5
258/258 ━━━━━━━━━━━━━━━━━━━━ 162s 626ms/step - accuracy: 0.8415 - loss: 0.4132 - val_accuracy: 0.8363 - val_loss: 0.4648
Epoch 3/5
258/258 ━━━━━━━━━━━━━━━━━━━━ 166s 644ms/step - accuracy: 0.8651 - loss: 0.3446 - val_accuracy: 0.8246 - val_loss: 0.4510
Epoch 4/5
258/258 ━━━━━━━━━━━━━━━━━━━━ 211s 679ms/step - accuracy: 0.8856 - loss: 0.2995 - val_accuracy: 0.8331 - val_loss: 0.4981
Epoch 5/5
258/258 ━━━━━━━━━━━━━━━━━━━━ 197s 660ms/step - accuracy: 0.9049 - loss: 0.2514 - val_accuracy: 0.8142 - val_loss: 0.5187


#**Model Evaluation**

In [25]:
loss, accuracy = model.evaluate(X_test, Y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

161/161 ━━━━━━━━━━━━━━━━━━━━ 10s 64ms/step - accuracy: 0.8135 - loss: 0.5294
Test Loss: 0.5293667316436768
Test Accuracy: 0.8134835958480835


##**Building a Predictive System**

In [26]:
def predict_sentiment(review):
    # tokenize and pad the review
    sequence = tokenizer.texts_to_sequences([review])
    padded_sequence = pad_sequences(sequence, maxlen=200)

    # prediction
    prediction = model.predict(padded_sequence, verbose=0)

    # get class with highest probability
    predicted_class = prediction.argmax(axis=1)[0]

    # map class to sentiment
    labels = ["negative", "neutral", "positive"]

    return labels[predicted_class]

In [27]:
new_review = "Very disappointed with the quality, not worth the price."
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is: {sentiment}")

The sentiment of the review is: negative


In [28]:
new_review = "Amazing quality and works perfectly as described"
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is: {sentiment}")

The sentiment of the review is: positive


In [29]:
new_review = "The product is okay, not great but not bad either."
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is: {sentiment}")

The sentiment of the review is: neutral


In [30]:
# save model
model.save("sentiment_model.h5")

# save tokenizer
import pickle
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
    from google.colab import files

files.download("sentiment_model.h5")
files.download("tokenizer.pkl")

ModuleNotFoundError: No module named 'google.colab'

In [31]:
import pickle

# save tokenizer
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("✅ Sentiment tokenizer saved")

# model already saved
# model.save("sentiment_model.keras")
print("✅ Sentiment model saved")

✅ Sentiment tokenizer saved
✅ Sentiment model saved


In [32]:
import joblib
import pandas as pd
from scipy.sparse import load_npz

tfidf = joblib.load("tfidf_vectorizer.pkl")
nn_model = joblib.load("content_nn_model.pkl")

product_df = pd.read_csv("product_df.csv")

cf_model = joblib.load("cf_model.pkl")
user_item_sparse = load_npz("user_item_sparse.npz")

print("✅ All artifacts loaded correctly")

C:\Users\srishti.v.singh\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.8.0 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\srishti.v.singh\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.8.0 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\srishti.v.singh\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWa

✅ All artifacts loaded correctly


C:\Users\srishti.v.singh\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator NearestNeighbors from version 1.8.0 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [34]:
import joblib
import pandas as pd
from scipy.sparse import load_npz

tfidf = joblib.load("tfidf_vectorizer.pkl")
nn_model = joblib.load("content_nn_model.pkl")

product_df = pd.read_csv("product_df.csv")

cf_model = joblib.load("cf_model.pkl")
user_item_sparse = load_npz("user_item_sparse.npz")

print("✅ Everything loaded successfully")

C:\Users\srishti.v.singh\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.8.0 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\srishti.v.singh\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.8.0 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


✅ Everything loaded successfully


C:\Users\srishti.v.singh\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator NearestNeighbors from version 1.8.0 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
